In [2]:
%pip install groq fastapi uvicorn

/home/samir/Desktop/RAG_TEST/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
 # uncomment if not installed

import os
import json
import time
import threading
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse, JSONResponse, StreamingResponse

# ---------------------------------------------------------------------
# MEMORY -- the single source of truth for the whole chatbot.
# Every user message, assistant reply, and (later) tool call/result gets
# appended here. The chat UI only ever displays "user" and "assistant"
# turns that have text -- everything else stays invisible to the user.
# ---------------------------------------------------------------------
messages = [
    {"role": "system", "content": "You are a helpful assistant."}
]


def add_message(role, content, **extra):
    msg = {"role": role, "content": content, **extra}
    messages.append(msg)
    return msg


def visible_messages():
    """What the browser is allowed to see: text-bearing user/assistant turns."""
    return [
        {"role": m["role"], "content": m["content"]}
        for m in messages
        if m["role"] in ("user", "assistant") and m.get("content")
    ]


# ---------------------------------------------------------------------
# THE "BRAIN SLOT" -- every later cell in this notebook REPLACES this
# function. The FastAPI route below always calls whatever RESPOND
# currently points to, so re-running a cell instantly upgrades the
# live UI. No server restart required.
# ---------------------------------------------------------------------
def RESPOND():
    """Level 0: memory only. Nothing streams back -- there's no brain yet."""
    yield ""


# ---------------------------------------------------------------------
# The chat webpage itself -- plain HTML + JS, zero external dependencies.
# ---------------------------------------------------------------------
CHAT_HTML = """<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>Tool Calling Demo Chat</title>
<style>
  * { box-sizing: border-box; }
  body {
    margin: 0; font-family: -apple-system, Segoe UI, Roboto, sans-serif;
    background: #0f1115; color: #e6e6e6; height: 100vh; display: flex; flex-direction: column;
  }
  header { padding: 14px 20px; border-bottom: 1px solid #23262e; font-weight: 600; }
  #chat { flex: 1; overflow-y: auto; padding: 20px; display: flex; flex-direction: column; gap: 12px; }
  .bubble { max-width: 70%; padding: 10px 14px; border-radius: 14px; line-height: 1.45; white-space: pre-wrap; }
  .user { align-self: flex-end; background: #2f6feb; color: white; border-bottom-right-radius: 4px; }
  .assistant { align-self: flex-start; background: #1c1f27; border: 1px solid #2a2e38; border-bottom-left-radius: 4px; }
  .assistant.pending { color: #8b90a0; font-style: italic; }
  .assistant code { background: #11131a; padding: 1px 5px; border-radius: 4px; font-size: 0.9em; }
  .assistant pre { background: #11131a; padding: 10px; border-radius: 8px; overflow-x: auto; }
  #inputRow { display: flex; gap: 10px; padding: 14px 20px; border-top: 1px solid #23262e; }
  #textInput { flex: 1; padding: 10px 14px; border-radius: 10px; border: 1px solid #2a2e38; background: #1c1f27; color: #e6e6e6; font-size: 14px; }
  #sendBtn { padding: 10px 18px; border-radius: 10px; border: none; background: #2f6feb; color: white; font-weight: 600; cursor: pointer; }
  #sendBtn:disabled { opacity: 0.5; cursor: not-allowed; }
</style>
</head>
<body>
  <header>Tool Calling Demo Chat</header>
  <div id="chat"></div>
  <div id="inputRow">
    <input id="textInput" placeholder="Type a message and hit Enter..." autocomplete="off">
    <button id="sendBtn">Send</button>
  </div>
<script>
const chatEl = document.getElementById('chat');
const inputEl = document.getElementById('textInput');
const sendBtn = document.getElementById('sendBtn');

function renderMD(text) {
  const esc = text.replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');
  return esc
    .replace(/```([\\s\\S]*?)```/g, '<pre><code>$1</code></pre>')
    .replace(/`([^`]+)`/g, '<code>$1</code>')
    .replace(/\\*\\*([^*]+)\\*\\*/g, '<strong>$1</strong>')
    .replace(/\\*([^*]+)\\*/g, '<em>$1</em>')
    .replace(/\\n/g, '<br>');
}

function addBubble(role, text) {
  const div = document.createElement('div');
  div.className = 'bubble ' + role;
  div.innerHTML = renderMD(text);
  chatEl.appendChild(div);
  chatEl.scrollTop = chatEl.scrollHeight;
  return div;
}

async function loadHistory() {
  const res = await fetch('/api/messages');
  const msgs = await res.json();
  chatEl.innerHTML = '';
  msgs.forEach(m => addBubble(m.role, m.content));
}

async function sendMessage() {
  const text = inputEl.value.trim();
  if (!text) return;
  inputEl.value = '';
  sendBtn.disabled = true;

  addBubble('user', text);
  const pending = addBubble('assistant', '...');
  pending.classList.add('pending');

  const res = await fetch('/api/stream', {
    method: 'POST',
    headers: {'Content-Type': 'application/json'},
    body: JSON.stringify({text})
  });

  const reader = res.body.getReader();
  const decoder = new TextDecoder();
  let full = '';
  while (true) {
    const {done, value} = await reader.read();
    if (done) break;
    full += decoder.decode(value, {stream: true});
    pending.classList.remove('pending');
    pending.innerHTML = renderMD(full || '...');
    chatEl.scrollTop = chatEl.scrollHeight;
  }

  // Resync fully with backend memory (source of truth) once streaming ends.
  await loadHistory();
  sendBtn.disabled = false;
  inputEl.focus();
}

sendBtn.addEventListener('click', sendMessage);
inputEl.addEventListener('keydown', e => { if (e.key === 'Enter') sendMessage(); });

loadHistory();
</script>
</body>
</html>
"""

# ---------------------------------------------------------------------
# FastAPI server -- started ONCE, in a background thread.
# ---------------------------------------------------------------------
app = FastAPI()


@app.get("/")
async def index():
    return HTMLResponse(CHAT_HTML)


@app.get("/api/messages")
async def get_messages():
    return JSONResponse(visible_messages())


@app.post("/api/stream")
async def stream(request: Request):
    data = await request.json()
    user_text = data.get("text", "")
    add_message("user", user_text)

    def gen():
        # RESPOND is a plain sync generator -- Starlette runs it in a
        # threadpool under the hood, so it won't block the event loop.
        yield from RESPOND()

    return StreamingResponse(gen(), media_type="text/plain")


PORT = 8765


def run_server():
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")


if not globals().get("_SERVER_STARTED"):
    threading.Thread(target=run_server, daemon=True).start()
    _SERVER_STARTED = True
    time.sleep(1.5)  # give uvicorn a moment to actually bind the port

# ---------------------------------------------------------------------
# Colab runs this notebook on a REMOTE machine -- "127.0.0.1" there is
# NOT your laptop's localhost, so your browser can never reach it
# directly. Colab provides a built-in proxy for exactly this case.
# Everywhere else (local Jupyter, VS Code), plain localhost works fine.
# ---------------------------------------------------------------------
try:
    from google.colab.output import eval_js  # only importable inside Colab
    url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    print(f"Running in Google Colab. Chat UI: {url}")
except ImportError:
    print(f"Chat UI running at http://127.0.0.1:{PORT}")

print("Open that URL in a browser tab and keep it open -- later cells upgrade it live.")


ModuleNotFoundError: No module named 'uvicorn'

## 2️⃣ Brain — connect Groq, with real streaming

We now **reassign** `RESPOND`. The server is already running — it never restarts. The very next message you send in the browser tab will use this new version, because the FastAPI route always calls the current global `RESPOND`.

Go to the browser and send a message. You'll see it reply for real, token by token.


In [ ]:
from groq import Groq


def get_secret(name):
    """Colab Secrets are NOT auto-injected into os.environ -- they must be
    fetched explicitly via google.colab.userdata. Falls back to a plain
    environment variable everywhere else (local Jupyter, VS Code, etc.)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except ImportError:
        return os.environ.get(name)


client = Groq(api_key=get_secret("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"


def RESPOND():
    """Level 1: the brain is connected. Streams a real LLM reply -- no tools yet."""
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": m["role"], "content": m["content"]} for m in messages],
        stream=True,
    )
    full_text = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        if delta:
            full_text += delta
            yield delta
    add_message("assistant", full_text)


print("Brain connected. Go to the browser tab and send a message -- it replies for real now.")


## 🚫 The brain can talk, but it can't *do* anything

In the browser, ask something that needs a real action or live data — e.g. *"What's the weather in Pune right now?"* or *"What's 482 × 17?"* Watch it either refuse, guess, or hallucinate a plausible-sounding but made-up answer.

That's expected: an LLM only ever **predicts the next token of text**. It has no hands, no internet connection, no calculator. To let it actually *act*, we have to give it tools.


## 3️⃣ Add real functions — but the model doesn't know they exist yet

Let's write actual Python functions that *can* get things done. `RESPOND` is untouched by this cell — still Level 1, no `tools=` passed anywhere. Defining a Python function in this notebook does not magically teach the model about it.

After running this cell, go to the browser and type: *"Please call the get_weather function for Pune."* It will **still fail** — the model has zero knowledge of what functions exist in our codebase unless we formally describe them to it.


In [2]:
def get_weather(city: str) -> str:
    """Pretend weather API -- swap this for a real API call if you like."""
    fake_db = {
        "pune": "28°C, partly cloudy",
        "mumbai": "31°C, humid",
        "delhi": "24°C, clear skies",
    }
    return fake_db.get(city.lower(), f"No weather data for {city}")


def calculator(expression: str) -> str:
    """Evaluates a basic arithmetic expression, e.g. '12 * (4 + 3)'."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"


print("Functions defined. RESPOND is unchanged -- go try asking for the weather in the browser, it still fails.")


Functions defined. RESPOND is unchanged -- go try asking for the weather in the browser, it still fails.


## 4️⃣ Register the tools — and watch the model *ask* instead of *answer*

We describe each function as a JSON Schema (`name`, `description`, `parameters`) and pass that list as `tools=`. This is the menu of actions the model is allowed to request. Nothing outside this menu can ever be called.

This version of `RESPOND` deliberately stops at the request — it does **not** execute anything yet, just surfaces the raw tool call so you can see it. In the browser, ask for the weather again. Instead of an answer, you'll see a bubble like:

> 🔧 Model wants to call **get_weather** with args `{"city": "Pune"}` — (not executed yet, see next cells)


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. Pune"}
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a basic arithmetic expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "e.g. '12 * (4 + 3)'"}
                },
                "required": ["expression"],
            },
        },
    },
]


def _trimmed(m):
    """Only forward the fields the Groq API accepts."""
    keys = ("role", "content", "tool_calls", "tool_call_id", "name")
    return {k: m[k] for k in keys if k in m}


def RESPOND():
    """Level 2: tools registered. The model can now ASK to call one -- but
    we deliberately do NOT execute it yet, just surface the raw request."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[_trimmed(m) for m in messages],
        tools=tools,
    )  # non-streaming on purpose, so we can inspect the whole tool_calls object at once
    msg = response.choices[0].message

    if msg.tool_calls:
        tc = msg.tool_calls[0]
        text = (
            f"🔧 Model wants to call **{tc.function.name}** "
            f"with args `{tc.function.arguments}` — (not executed yet, see next cells)"
        )
    else:
        text = msg.content

    add_message("assistant", text)
    yield text


print("Tools registered. Go ask for the weather again -- watch it ASK instead of ANSWER.")


## 5️⃣ Extract the function name and arguments

The arguments arrive as a JSON **string**, not a Python dict — we must parse them before we can call the real Python function with them. This cell runs a fresh, self-contained request (independent of whatever's in the live chat) just to inspect that raw object clearly.


In [ ]:
demo_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What's the weather in Pune?"},
]

demo_response = client.chat.completions.create(
    model=MODEL,
    messages=demo_messages,
    tools=tools,
)
tool_call = demo_response.choices[0].message.tool_calls[0]

function_name = tool_call.function.name
function_args = json.loads(tool_call.function.arguments)  # string -> dict

print("Model wants to call:", function_name)
print("With arguments:", function_args)


## 6️⃣ Execute the tool, send the result back, and complete the loop

The final version of `RESPOND`:

1. Asks the model what it wants to do
2. If it requests a tool call: look up the real Python function, run it with the extracted arguments, append the result to memory as a `role: "tool"` message linked via `tool_call_id`
3. Loop back to step 1 — now the model has real data and can write a normal, natural-language answer
4. If it returns plain text instead: stream that out and stop

Go to the browser and ask for the weather (or `15 * 12`) one more time — this time you get a real, correct answer.


In [ ]:
available_functions = {
    "get_weather": get_weather,
    "calculator": calculator,
}


def RESPOND():
    """Level 3: the full tool-calling loop. Keeps going until the model
    returns a plain-text answer -- this is what a production chatbot does."""
    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[_trimmed(m) for m in messages],
            tools=tools,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            add_message("assistant", msg.content)
            yield msg.content
            return

        # Record the request itself (content=None -> invisible in the UI,
        # but still part of memory so the model sees its own past turn).
        add_message(
            "assistant",
            None,
            tool_calls=[
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                }
                for tc in msg.tool_calls
            ],
        )

        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            result = available_functions[name](**args)  # <-- the real function actually runs
            add_message("tool", str(result), tool_call_id=tc.id, name=name)

        # loop again -- the model now has real data and can answer for real


print("Full loop wired up. Go ask for the weather (or a calculation) again -- now you get a real answer.")


## 🧠 Peek at memory vs. what the browser shows

Print `messages` below — this is everything the backend tracked across the whole conversation, including the invisible `tool_calls` request and the `role: "tool"` result. The browser only ever rendered the `user` bubbles and the final `assistant` text bubbles (via `/api/messages` → `visible_messages()`). Everything else was working quietly behind the scenes — that gap between the two is exactly what "tool calling" is.


In [ ]:
for m in messages:
    shown = {k: v for k, v in m.items() if k != "role"}
    print(f"{m.get('role'):9s} -> {shown}")
